In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_agent

In [3]:
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearch(max_results=3)

In [5]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("What is the latest news on AI?")
result

{'query': 'What is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://guides.library.georgetown.edu/ai/news',
   'title': 'Artificial Intelligence (Generative) Resources: Latest News on AI',
   'content': 'Search this Guide  Search \n\n# Artificial Intelligence (Generative) Resources: Latest News on AI\n\n   Home\n   AI Fundamentals\n   AI Tools for Research\n   How to Craft Prompts\n   Research Resources on AI\n   Latest News on AI\n       AI News from the New York Times, WSJ, and Washington Post (ProQuest)\n       AI News in Education Periodicals (EBSCO)\n\n   Ethics & AI\n   Citing Generative AI\n   AI, Authorship, & Copyright\n   Campus Resources and Policies\n\n## Publications covering AI [...] ## AI News from the New York Times, WSJ, and Washington Post (ProQuest) [...] Skip to Main Content\n## User account menu\n\nYour Account\n\nToggle navigation menuImage 1: Georgetown University Library Home\n\n   \n       All 

In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
)

In [10]:
response = llm.invoke("What year is it?")
response

AIMessage(content="It's 2026.", additional_kwargs={'reasoning_content': 'The user asks: "What year is it?" The system says current date is 2026-09-01. So answer: 2026.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 76, 'total_tokens': 122, 'completion_time': 0.047934832, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.003761002, 'prompt_tokens_details': None, 'queue_time': 0.156844258, 'total_time': 0.051695834}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05c45-2a55-7541-9823-8f504419d869-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 46, 'total_tokens': 122, 'output_token_details': {'reasoning': 32}})

In [11]:
# tools = []
tools = [search_tool, get_weather]

In [12]:
# CREATE AGENT

agent = create_agent(
    model = llm,
    tools = tools
)

In [13]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the current weather of Ongole"}
    ]
})

In [14]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 36 °C  
- **Condition:** Overcast  
- **Humidity:** 35%  

Feel free to let me know if you need any other details or forecasts!


In [15]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather of Ongole

City: Ongole
Temperature: 36°C
Weather: Overcast 
Humidity: 35%
**Current Weather in Ongole**

- **Temperature:** 36 °C  
- **Condition:** Overcast  
- **Humidity:** 35%  

Feel free to let me know if you need any other details or forecasts!


In [16]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about the Latest News on Iran and US War"}
    ]
})

In [17]:
for msg in response["messages"]:
    print(msg.content)

Tell me about the Latest News on Iran and US War

{"query": "latest news Iran US war", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://news.sky.com/story/iran-us-latest-trump-bessent-economic-d-day-13509565", "title": "Iran war latest: US 'can't get out' of conflict and 'has run ...", "content": "Skip to content\n\n# Iran war latest: US 'can't get out' of conflict and 'has run out of patience'\n\nTuesday 25 August 2026 01:21, UK\n\nVideo Player is loading.\n\nLoaded: 0%\n\nCurrent Time 0:00\n\nDuration 0:00\n\nRemaining Time -0:00\n\nStream Type LIVE\n\nThis is a modal window.\n\nBeginning of dialog window. Escape will cancel and close the window.\n\nEnd of dialog window.\n\nThis is a modal window. This modal can be closed by pressing the Escape key or activating the close button.\n\nUS launches 'Operation Economic Outcast' against Iran\n\nWhy you can trust Sky News\n\nListen to exclusive podcasts from Sky News\n\nSky News Insider\n\nSky News Ins